# 7장 · 멀티헤드 어텐션 — 여러 관점으로 동시에 🧠🧠🧠

웹 챕터 **"멀티헤드 어텐션"** 과 짝을 이루는 실습입니다.
웹에서 두 헤드의 집중을 봤다면, 여기서는 **여러 헤드를 계산하고 이어붙이기(concat)** 를 직접 합니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요
from tutor import chat

chat('ch7')

## 1. 두 헤드의 집중도

"강아지 · 공 · 물었다". 두 헤드가 각기 다른 관점(주어 vs 목적어)으로 집중합니다.
각 헤드는 ch6처럼 자기 Q·K·V 로 이 집중도를 계산하지만, 여기선 결과 행렬을 바로 씁니다. (웹과 같은 값)

In [1]:
import numpy as np

WORDS = ['강아지', '공', '물었다']
VALUE = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])

# 헤드별 집중도 (행=질문, 열=참고), 이미 소프트맥스된 값
HEAD1 = np.array([[0.6,0.2,0.2],[0.2,0.6,0.2],[0.65,0.15,0.2]])  # 주어에 민감
HEAD2 = np.array([[0.6,0.2,0.2],[0.2,0.6,0.2],[0.15,0.65,0.2]])  # 목적어에 민감

q = 2   # '물었다'
print('물었다 기준 집중도')
print(' 헤드1:', dict(zip(WORDS, HEAD1[q])))
print(' 헤드2:', dict(zip(WORDS, HEAD2[q])))

물었다 기준 집중도
 헤드1: {'강아지': np.float64(0.65), '공': np.float64(0.15), '물었다': np.float64(0.2)}
 헤드2: {'강아지': np.float64(0.15), '공': np.float64(0.65), '물었다': np.float64(0.2)}


## 2. 각 헤드의 출력 = 집중도로 Value 가중합

ch6에서 배운 대로, 헤드마다 집중도로 Value 를 섞어 출력 벡터를 냅니다.

In [2]:
out1 = HEAD1[q] @ VALUE
out2 = HEAD2[q] @ VALUE
print('헤드1 출력:', np.round(out1, 2), '(강아지 쪽으로 기움)')
print('헤드2 출력:', np.round(out2, 2), '(공 쪽으로 기움)')

헤드1 출력: [0.75 0.25] (강아지 쪽으로 기움)
헤드2 출력: [0.25 0.75] (공 쪽으로 기움)


## 3. 이어붙이기(concat) → 최종 혼합

두 출력을 나란히 이어붙이면 더 긴 벡터가 됩니다. 실제로는 그 뒤에 선형층 W_O 로 한 번 더 섞어요.
$$\text{MultiHead} = \text{Concat}(\text{head}_1, \dots)\, W_O$$

In [3]:
concat = np.concatenate([out1, out2])
print('concat:', np.round(concat, 2), '→ 숫자', len(concat), '개')

# 최종 선형층(예시 가중치). 4개 → 2개로 섞기
rng = np.random.default_rng(0)
W_O = rng.normal(size=(4, 2))
final = concat @ W_O
print('최종 출력(W_O로 섞은 뒤):', np.round(final, 2))

concat: [0.75 0.25 0.25 0.75] → 숫자 4 개
최종 출력(W_O로 섞은 뒤): [1.1  0.73]


## 4. 헤드가 많아도 계산량이 폭증하지 않는다

전체 차원을 헤드 수로 **나눠** 갖기 때문입니다. 각 헤드는 작은 차원만 담당해요.

In [4]:
d_model = 512
for h in [1, 2, 4, 8, 16]:
    print(f'헤드 {h:2d}개 → 헤드당 차원 {d_model // h}')

헤드  1개 → 헤드당 차원 512
헤드  2개 → 헤드당 차원 256
헤드  4개 → 헤드당 차원 128
헤드  8개 → 헤드당 차원 64
헤드 16개 → 헤드당 차원 32


## 5. 직접 해보기 ✏️

헤드1 출력 `(0.7, 0.3)`, 헤드2 출력 `(0.2, 0.8)` 를 이어붙이면 숫자가 몇 개일까요? (웹 문제와 동일)

In [5]:
a = np.array([0.7, 0.3]); b = np.array([0.2, 0.8])
c = np.concatenate([a, b])
print('concat:', c, '→ 길이', len(c))

assert len(c) == 4, '2 + 2 = 4 개여야 합니다!'
print('✅ [0.7, 0.3, 0.2, 0.8] (숫자 4개)')

concat: [0.7 0.3 0.2 0.8] → 길이 4
✅ [0.7, 0.3, 0.2, 0.8] — 숫자 4개


---
여러 관점(헤드)을 한 번에 잡아 이어붙였습니다. 어텐션 파트 완성! 🎉
다음 챕터(ch8)에서 이 어텐션을 **잔차연결·층정규화·피드포워드**와 묶어 **트랜스포머 블록**을 만듭니다. 🧱